# Coursework 2 Part 1
**Replace CID in the file name with your CID**

# Outline


- [Task 1](#task-1): Classification with a Convolutional Neural Network <a name="index-task-1"></a>
  - [(1.1)](#task-11) <a name="index-task-11"></a>
  - [(1.2)](#task-12) <a name="index-task-12"></a>
- [Task 2](#task-2): Dimensionality Reduction with Non-Negative Matrix Tri-Factorisation <a name="index-task-2"></a>
  - [(2.1)](#task-21) <a name="index-task-21"></a>
  - [(2.2)](#task-22)  <a name="index-task-22"></a>

In [5]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

# Importing losses, activation functions and layers from PyTorch
from torch.nn import Sequential, CrossEntropyLoss, Conv1d, MaxPool1d, Flatten, Linear, ReLU, Softmax, Parameter, Dropout
from torch.utils.data import TensorDataset, DataLoader

<a name="task-1"></a>

# Task 1: Classification with a Convolutional Neural Network [(index)](#index-task-1)

<a name="task-11"></a>

## (1.1) [(index)](#index-task-11)

In [13]:
data = pd.read_csv("data/neural_activity.csv")
data.head()

,Count at t=1,Count at t=2,Count at t=3,Count at t=4,Count at t=5,Count at t=6,Count at t=7,Count at t=8,Count at t=9,Count at t=10,...,Count at t=194,Count at t=195,Count at t=196,Count at t=197,Count at t=198,Count at t=199,Count at t=200,Brain Region Index,Trial Index,Trial Type
0,25,25,26,27,27,28,29,30,31,32,...,64,66,67,69,71,72,73,1,1,B
1,68,68,69,69,70,71,71,72,72,73,...,38,37,37,36,35,35,33,1,2,F
2,30,29,29,29,29,29,29,30,30,31,...,33,32,31,30,29,28,27,1,3,L
3,24,25,26,28,29,31,32,35,37,39,...,94,93,91,90,88,86,84,1,4,R
4,59,59,59,59,59,58,58,58,58,59,...,37,34,33,31,29,28,26,1,5,F


In [14]:
torch.manual_seed(10)

# This function is taken from CNN notebook, then modified
def get_model(x_train, n_filters, k, pool_size, stride_pool, classes):
    """
    CNN model in PyTorch:
    - Layers are Conv1d(+ReLU), MaxPool1d, Flatten and Linear(+Softmax).
    - It features an Adam optimiser and CrossEntropyLoss criterion.

    Parameters:
        x_train (np.ndarray): Training data
        n_filters (int): Number of filters to be used in the convolutional layer
        k (int): Kernel size in the convolutional layer
        pool_size (int): MaxPool1d window size
        stride_pool (int): Stride of the MaxPool1d sliding window
        classes (List): Output classes

    Returns:
        Model, criterion and optimiser.

    """

    l_out_conv = x_train.shape[2] - k + 1 # Length after Conv1d (note that the stride is 1) ## <-- EDIT THIS LINE
    l_out_pool = int(np.floor((l_out_conv - pool_size)/2)) + 1 # Length after MaxPool1d ## <-- EDIT THIS LINE
    l_in_linear = l_out_pool * n_filters # Size before Linear layer ## <-- EDIT THIS LINE
    np.testing.assert_allclose(l_in_linear, 392)

    model = Sequential(
        Conv1d(x_train.shape[1], n_filters, kernel_size=k),
        ReLU(), ## <-- SOLUTION
        MaxPool1d(kernel_size=pool_size, stride=stride_pool), ## <-- SOLUTION
        Flatten(), ## <-- SOLUTION
        Dropout(0.8),
        Linear(l_in_linear, len(classes)), ## <-- SOLUTION
    )

    criterion = CrossEntropyLoss() ## <-- EDIT THIS LINE
    optimiser = torch.optim.Adam(model.parameters()) ## <-- EDIT THIS LINE

    return model, criterion, optimiser

In [ ]:
# Reshape data into N_trials by N_regions by p = 200 (length of time series)

N_regions = len(data["Brain Region Index"].unique())
N_trials = len(data["Trial Index"].unique())
p = 200

count_data = data.drop(columns=[
    "Brain Region Index", 
    "Trial Index", 
    "Trial Type"])
data_arr = count_data.to_numpy()
tensor = data_arr.reshape(((N_regions, N_trials, p))).T

y = data["Trial Type"].to_numpy()

test_idx = [56, 30, 42, 63, 7, 66, 2, 77, 78, 13, 39, 61, 5, 67, 70, 46]

# Get train indices
all_idx = np.arange(len(tensor))
train_idx = np.setdiff1d(all_idx, test_idx)

# Divide data
X_train = tensor[train_idx]
X_test = tensor[test_idx]

y_train = y[train_idx]
y_test = y[test_idx]

<a name="task-12"></a>

## (1.2) [(index)](#index-task-12)

<a name="task-2"></a>

# Task 2: Dimensionality Reduction with Non-Negative Matrix Tri-Factorisation [(index)](#index-task-2)

<a name="task-21"></a>

## (2.1) [(index)](#index-task-21)

<a name="task-22"></a>

## (2.2) [(index)](#index-task-22)